# k8c — Fine-tune reranker (GPU T4 ×2): bge (GPU 0) và Qwen3-Reranker-4B + LoRA (GPU 1) song song

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On.
**Add Input:** `mir2-code`, `mir2-data`, mọi output k8b (`synth/labels-*.parquet`, `synth/pairs.parquet`, `qexp/synth`).
Loss: chưng cất theo danh sách (KL tới softmax(3 × điểm teacher 0–2) trên ứng viên của chính câu + 2 đoạn câu khác) +
BCE kéo điểm về nhãn mềm của teacher. Mỗi câu tổng hợp một nhóm (bge 16 đoạn, qwen 8 đoạn).
5% câu tổng hợp để kiểm tra: in AUC / độ chính xác cặp TRƯỚC và SAU khi train (sau phải cao hơn).
**Đầu ra:** `ft/bge/` (model đầy đủ), `ft/qwen/` (adapter LoRA) → *New Dataset* **`mir2-ft`** → k5 với
`CROSS_MODEL = 'bge'` / `QWEN_LORA = 'qwen'` (không dùng REUSE cho phương pháp đã fine-tune).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k8c-v3 (15/10/2026)'
TRAIN = ['bge', 'qwen']              # bỏ bớt nếu chỉ train một model
MAX_GROUPS = {'bge': 0, 'qwen': 3000}   # 0 = mọi nhóm; qwen ~3 s/nhóm 8 đoạn trên T4
EPOCHS = 1
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
def _ver(h):                                                  # output notebook cũ (vd mir2-chunks/code) cũng có mirret/
    v = os.path.join(os.path.dirname(os.path.dirname(h)), 'VERSION.txt')
    return open(v, encoding='utf-8').read().strip() if os.path.exists(v) else ''
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True),
              key=lambda h: (_ver(h), -len(h)), reverse=True)   # bản mã mới nhất theo VERSION.txt (ngày giờ)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=())

In [ ]:
# (2) Gom nhãn của mọi phần k8b
os.makedirs(f'{K.work}/synth', exist_ok=True)
for d in locate_all('synth/pairs.parquet'):
    for f in glob.glob(f'{d}/synth/labels*.parquet') + [f'{d}/synth/pairs.parquet']:
        shutil.copy(f, f'{K.work}/synth/')
for d in locate_all('qexp/synth/queries_vi.parquet'):
    shutil.copytree(f'{d}/qexp/synth', f'{K.work}/qexp/synth', dirs_exist_ok=True)
print(sorted(os.listdir(f'{K.work}/synth')))
sh('python -m pip install -q peft')
sh('python -m pip uninstall -y -q torchao')           # peft mới đòi torchao > 0.16 (Kaggle có 0.10) — không dùng tới

In [ ]:
# (3) Train song song: mỗi model một GPU
cmds = []
for g, kind in enumerate(TRAIN):
    mg = f' --max-groups {MAX_GROUPS[kind]}' if MAX_GROUPS.get(kind) else ''
    cmds.append(f'CUDA_VISIBLE_DEVICES={g} python -m mirret.finetune {kind} --config {K.cfg} --out {K.work}/ft/{kind} '
                f'--device cuda:0 --epochs {EPOCHS}{mg}')
par(cmds, 'finetune')
for kind in TRAIN:
    print(kind, open(f'{K.work}/ft/{kind}/finetune_meta.json', encoding='utf-8').read())
report('k8c')